# Session 4, The Challenger launch decision

> **Whole-class build.** It is the night of 27 January 1986. The forecast for launch time is **31°F**, colder than any launch before. You have the data from the 23 previous flights. **Should the shuttle fly?**

Today you are not just fitting a model. You are the analyst who has to turn that model into a **recommendation**, and live with it.

**How we'll work**
- Let Claude write the code. *You* read the model and make the call.
- Predict before you compute. Commit to a view, then see if the data agrees.

In [ ]:
# --- Load the data -------------------------------------------------
# The data loads from the course's public notebook repo, nothing to upload.
# No internet? Set DATA_URL = "" and upload the file when prompted (Colab only).
DATA_URL = "https://raw.githubusercontent.com/eoinlane/BUU33803-student-notebooks/main/session4/data/challenger.csv"

import pandas as pd

if DATA_URL:
    df = pd.read_csv(DATA_URL)
else:
    try:
        from google.colab import files
        print("Upload challenger.csv ...")
        up = files.upload()
        df = pd.read_csv(next(iter(up)))
    except ImportError:
        df = pd.read_csv("data/challenger.csv")   # local fallback (repo: data/challenger.csv)

df.head()

## What the columns mean

Each row is one previous shuttle flight (23 in total, every flight whose boosters were recovered).

| Column | Meaning |
|---|---|
| `flight` | mission code |
| `temp_f` | air temperature at launch, °F |
| `damage` | 1 = O-ring damage occurred (erosion or blow-by), 0 = no damage |

*Where the data comes from: the UCI Machine Learning Repository's Challenger O-ring data (Draper 1993, after Dalal, Fowlkes & Hoadley 1989), the same flight record the engineers had.*

## Step 1, Predict before you compute  🎯

Before any modelling, commit to a view (write it in the next cell):

- The coldest previous launch was 53°F. The forecast for launch is **31°F**.
- On a scale from *unlikely* to *almost certain*, how likely is O-ring damage at 31°F?
- Would you launch?

In [ ]:
# Your view before you model (just notes):
#
# How likely is damage at 31F?  :
# Launch or delay?              :

## Step 2, Look at the data first

> Ask Claude: *"Plot damage (0/1) against launch temperature. Colour the damage points differently and mark 31°F."*

Before fitting anything, just look. Where do the failures sit?

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 3.5))
plt.scatter(df['temp_f'], df['damage'], s=60,
            c=df['damage'].map({0: 'navy', 1: 'firebrick'}))
plt.axvline(31, color='orange', ls='--', label='launch forecast: 31°F')
plt.yticks([0, 1], ['no damage', 'damage'])
plt.xlabel('Temperature at launch (°F)'); plt.legend(); plt.show()

**Notice** what almost sank the real decision: the cold launches are exactly where the damage clusters, but 31°F sits far to the *left* of anything we have ever observed.

**Now see it the way the room did.** The chart argued over the night before left out the flights with *no* damage. Plot only the damaged flights:

In [ ]:
damaged = df[df['damage'] == 1]
plt.figure(figsize=(7, 2.2))
plt.scatter(damaged['temp_f'], damaged['damage'], s=60, c='firebrick')
plt.yticks([1], ['damage']); plt.xlim(30, 85)
plt.xlabel('Temperature at launch (°F), damaged flights only'); plt.show()

Damage at 53°F and at 75°F: on this chart temperature looks like it doesn't matter. The pattern only appears when you add back the **16 flights that had no damage**, almost all of them warm. Leaving out the 'nothing happened' cases hid the trend.

## Step 3, Fit the logistic model (let Claude drive)

> Ask Claude: *"Fit a logistic regression of damage on temp_f using statsmodels and print the summary. Then give me the predicted probability of damage at 31°F."*

In [ ]:
import statsmodels.formula.api as smf
import numpy as np

model = smf.logit('damage ~ temp_f', data=df).fit()
print(model.summary())

p31 = model.predict(pd.DataFrame({'temp_f': [31]}))[0]
print(f'\nP(O-ring damage at 31°F) = {p31:.3f}')

The coefficient on `temp_f` is **negative**: colder → more likely to fail. Ask Claude to turn it into an **odds ratio** and say it in one plain sentence (*"each 1°F drop multiplies the odds of damage by …"*). That sentence is what a manager actually needs.

## Step 4, The whole curve, and the honest caveat

> Ask Claude: *"Plot the fitted logistic curve from 30°F to 85°F over the data points, and shade the region colder than the coldest observed launch."*

In [ ]:
xs = np.linspace(30, 85, 200)
ps = model.predict(pd.DataFrame({'temp_f': xs}))

plt.figure(figsize=(7, 3.8))
plt.axvspan(30, df['temp_f'].min(), color='firebrick', alpha=0.07)
plt.plot(xs, ps, color='navy', lw=2.5)
plt.scatter(df['temp_f'], df['damage'], s=55,
            c=df['damage'].map({0: 'navy', 1: 'firebrick'}), zorder=5)
plt.axvline(31, color='orange', ls='--')
plt.xlabel('Temperature (°F)'); plt.ylabel('P(damage)'); plt.show()

**The caveat you must state:** 31°F is well outside the data (the shaded zone). The model is *extrapolating*. That is a real limitation, and yet every honest reading of the curve points the same way. Naming the uncertainty **and** still making the call is the mark of a good analyst.

## Step 5, How sure is the number?

Last week's third question, asked of this model. We have only 23 flights, so 0.9996 is one sample's answer. Do what we did to the Camac slope in Session 3: **redraw the 23 flights 2,000 times, refit, and keep each prediction at 31°F.**

> Ask Claude: *"Bootstrap the logistic model 2,000 times and show the spread of the predicted probability of damage at 31°F."*

In [ ]:
rng = np.random.default_rng(0)
preds = []
for _ in range(2000):
    redraw = df.sample(len(df), replace=True, random_state=rng.integers(1_000_000_000))
    if redraw['damage'].nunique() < 2:
        continue                      # a redraw with no damaged flights can't be fitted
    try:
        m = smf.logit('damage ~ temp_f', data=redraw).fit(disp=0, maxiter=200)
        preds.append(m.predict(pd.DataFrame({'temp_f': [31]}))[0])
    except Exception:
        pass

preds = np.array(preds)
print(f'{len(preds)} redraws fitted')
print(f'97.5% of them give P(damage at 31°F) of at least {np.percentile(preds, 2.5):.2f}')

The extrapolation is real, so the number is less certain than 0.9996 makes it look. But even the **cautious end** of the range says the same thing. That sentence belongs in your memo: it names the uncertainty *and* shows why the decision still holds.

## Step 6, Write the memo (this is the point)

You are on the 6pm call. Management wants **one paragraph**: launch or delay, in language a non-technical manager can act on, citing your model.

A strong memo:
1. **Leads with the decision**, don't bury it.
2. **Translates the number**, "near-certain seal damage at this temperature", not a raw coefficient.
3. **Names the uncertainty**, the extrapolation, then says why the recommendation still holds.

> Try it with Claude, then rewrite it in *your* voice. The judgement has to be yours.

In [ ]:
# Your one-paragraph recommendation to NASA management:
#

## Before you leave

1. In one sentence: what did the *clear* model show that a cluttered chart could hide?
2. The real engineers had this data. Why is **how you present evidence** part of the analysis, not an afterthought?
3. Keep your memo, you'll hand it in with your German credit answer.